In [0]:
%pip install yfinance faker

In [0]:
import random
from datetime import datetime, timedelta
import yfinance as yf
from faker import Faker
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType,
    DateType, DoubleType, LongType, TimestampType
)

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS wealth_management")
spark.sql("USE wealth_management")

In [0]:
TICKERS = ["AAPL", "MSFT", "NVDA", "JPM", "AMZN"]
NUM_CLIENTS = 25
NUM_TRANSACTIONS = 200

# ---------------------------------------------------------
# 1. Market Data & Asset Dimension Ingestion
# ---------------------------------------------------------
assets_data = []
prices_data = []

In [0]:
for idx, ticker in enumerate(TICKERS):
    asset_id = f"AST-{1000 + idx}"
    assets_data.append((asset_id, ticker, f"{ticker} Corp", "Equity", "USD", True))

    stock = yf.Ticker(ticker)
    df_yf = stock.history(period="1y").reset_index()
    display(df_yf.head())

    for _, row in df_yf.iterrows():
        prices_data.append((
            asset_id,
            row["Date"].date(),
            float(row["Open"]),
            float(row["Close"]),
            float(row["Close"]),
            int(row["Volume"]),
            datetime.now()
        ))



In [0]:
asset_schema = StructType([
    StructField("asset_id", StringType(), True),
    StructField("ticker", StringType(), True),
    StructField("asset_name", StringType(), True),
    StructField("asset_class", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("is_active", BooleanType(), True)
])

price_schema = StructType([
    StructField("asset_id", StringType(), True),
    StructField("price_date", DateType(), True),
    StructField("open_price", DoubleType(), True),
    StructField("close_price", DoubleType(), True),
    StructField("adjusted_close", DoubleType(), True),
    StructField("volume", LongType(), True),
    StructField("created_at", TimestampType(), True)
])

spark.createDataFrame(assets_data, schema=asset_schema).write.format("delta").mode("overwrite").saveAsTable("dim_asset")
spark.createDataFrame(prices_data, schema=price_schema).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim_market_prices")


print(f"Ingested {len(assets_data)} assets and {len(prices_data)} market price records.")

In [0]:
# ---------------------------------------------------------
# 2. Client Metadata Ingestion
# ---------------------------------------------------------
fake = Faker()
clients_data = []
client_ids = []
tiers = ["Retail", "HNW", "Ultra HNW"]
risks = ["Conservative", "Moderate", "Aggressive"]

for i in range(NUM_CLIENTS):
    c_id = f"CLT-{5000 + i}"
    client_ids.append(c_id)
    clients_data.append((
        c_id,
        fake.first_name(),
        fake.last_name(),
        random.choice(tiers),
        random.choice(risks),
        datetime.now()
    ))

In [0]:
client_schema = StructType([
    StructField("client_id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("client_tier", StringType(), True),
    StructField("risk_tolerance", StringType(), True),
    StructField("created_at", TimestampType(), True)
])

spark.createDataFrame(clients_data, schema=client_schema).write.format("delta").mode("overwrite").option("overwriteSchema",True).saveAsTable("dim_client")
print(f"Ingested {len(clients_data)} client profiles.")

In [0]:
# ---------------------------------------------------------
# 3. Transaction Ledger Ingestion
# ---------------------------------------------------------
txn_data = []
start_date = datetime.now() - timedelta(days=300)

for i in range(NUM_TRANSACTIONS):
    t_id = f"TXN-{10000 + i}"
    c_id = random.choice(client_ids)
    a_id = random.choice([a[0] for a in assets_data])
    t_date = start_date + timedelta(days=random.randint(0, 290))
    t_type = random.choice(["BUY", "SELL"])
    qty = round(random.uniform(10, 500), 4)
    price = round(random.uniform(50, 400), 4)
    gross = round(qty * price, 4)
    fee = round(random.uniform(5, 25), 2)
    net = gross + fee if t_type == "BUY" else gross - fee

    txn_data.append((
        t_id, c_id, a_id, t_date, t_type,
        qty, price, gross, fee, net,
        "CORE_LEDGER", datetime.now()
    ))

In [0]:
txn_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("client_id", StringType(), True),
    StructField("asset_id", StringType(), True),
    StructField("trade_date", TimestampType(), True),
    StructField("transaction_type", StringType(), True),
    StructField("quantity", DoubleType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("gross_amount", DoubleType(), True),
    StructField("brokerage_fee", DoubleType(), True),
    StructField("net_amount", DoubleType(), True),
    StructField("source_system", StringType(), True),
    StructField("ingested_at", TimestampType(), True)
])

spark.createDataFrame(txn_data, schema=txn_schema).write.format("delta").mode("overwrite").option("overwriteSchema",True).saveAsTable("fact_transactions")
print(f"Ingested {len(txn_data)} transactions into Delta Lake.")

In [0]:
spark.table("fact_transactions").show()

In [0]:
spark.sql('''select * from dim_market_prices''').show()

In [0]:
import json, time, math
from datetime import datetime
import yfinance as yf

# Map tickers to asset IDs upfront
ticker_map = {
    ticker: f"AST-{1000 + idx}" for idx, ticker in enumerate(TICKERS)
}

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.raw_landing;


In [0]:
dbutils.fs.mkdirs("/Volumes/workspace/default/raw_landing/ticks")

In [0]:
display(ticks)